In [18]:
import random
import time


strings = [
    "RED ZONE",
    "RED ZNOE",
    "NO PARKING",
    "NO PRKING",
    "METER EXPIRED",
    "METER EXPIRD"
]
pairs = [
    (
        random.choice(strings),
        random.choice(strings)
    )
    for _ in range(20_000)
]

In [19]:
import numpy as np
import pandas as pd
from rapidfuzz.fuzz import ratio
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

a_vals = [a for a, _ in pairs]
b_vals = [b for _, b in pairs]

start = time.perf_counter()

rapid_score = np.fromiter((ratio(a, b) for a, b in pairs), dtype=float, count=len(pairs))

emb_a = model.encode(a_vals, convert_to_numpy=True, normalize_embeddings=True, batch_size=64)
emb_b = model.encode(b_vals, convert_to_numpy=True, normalize_embeddings=True, batch_size=64)

mini_score = np.sum(emb_a * emb_b, axis=1) * 100
if_close = ((rapid_score > 75) | (mini_score > 85)).astype(int)

end = time.perf_counter()
print(end - start)

df = pd.DataFrame({
    "string_A": a_vals,
    "string_B": b_vals,
    "if_close": if_close,
    "minilm_score": np.round(mini_score, 2),
    "rapid_score": np.round(rapid_score, 2),
})

df.to_csv("comparison_scores_LLM.csv", index=False)
print(df.head(20))

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 5657.44it/s]


10.662448209011927
         string_A       string_B  if_close  minilm_score  rapid_score
0        RED ZNOE       RED ZONE         1     40.730000        87.50
1        RED ZNOE       RED ZNOE         1    100.000000       100.00
2      NO PARKING   METER EXPIRD         0      0.460000        27.27
3        RED ZNOE      NO PRKING         0     21.980000        23.53
4   METER EXPIRED  METER EXPIRED         1    100.000000       100.00
5        RED ZONE  METER EXPIRED         0     11.500000        28.57
6      NO PARKING       RED ZNOE         0      6.510000        22.22
7        RED ZONE     NO PARKING         0     21.080000        22.22
8   METER EXPIRED     NO PARKING         0      9.260000        26.09
9        RED ZONE  METER EXPIRED         0     11.500000        28.57
10       RED ZONE  METER EXPIRED         0     11.500000        28.57
11     NO PARKING       RED ZNOE         0      6.510000        22.22
12  METER EXPIRED   METER EXPIRD         1     77.760002        96.00
1

In [12]:
import random
import pandas as pd
from rapidfuzz.fuzz import ratio
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


model = SentenceTransformer("all-MiniLM-L6-v2")

results = []

start = time.perf_counter()

for a, b in pairs:

    rapid_score = round(ratio(a, b), 2)

    emb = model.encode([a, b])
    mini_score = round(
        cosine_similarity([emb[0]], [emb[1]])[0][0] * 100,
        2
    )

    if_close = 1 if (rapid_score > 75 or mini_score > 85) else 0

    results.append([
        a,
        b,
        if_close,
        mini_score,
        rapid_score
    ])

end = time.perf_counter()

print(end - start)

df = pd.DataFrame(results, columns=[
    "string_A",
    "string_B",
    "if_close",
    "minilm_score",
    "rapid_score"
])

df.to_csv("comparison_scores_LLM.csv", index=False)

print(df.head(20))

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4847.33it/s]


12.572339792008279
         string_A       string_B  if_close  minilm_score  rapid_score
0       NO PRKING       RED ZNOE         0     21.980000        23.53
1    METER EXPIRD       RED ZONE         0     10.810000        30.00
2        RED ZONE      NO PRKING         0     24.320000        23.53
3      NO PARKING      NO PRKING         1     30.420000        94.74
4        RED ZONE  METER EXPIRED         0     11.500000        28.57
5    METER EXPIRD   METER EXPIRD         1    100.000000       100.00
6      NO PARKING      NO PRKING         1     30.420000        94.74
7    METER EXPIRD     NO PARKING         0      0.460000        27.27
8      NO PARKING      NO PRKING         1     30.420000        94.74
9    METER EXPIRD  METER EXPIRED         1     77.760002        96.00
10  METER EXPIRED       RED ZONE         0     11.500000        28.57
11   METER EXPIRD      NO PRKING         0      9.210000        28.57
12   METER EXPIRD     NO PARKING         0      0.460000        27.27
1